# 09A · Disaggregated error analysis with Fairlearn

**Objective (15 min):** a support-ticket *escalation* classifier is used across two interaction
languages. The synthetic data is built so recall is worse for one group and aggregate accuracy hides
it. Use Fairlearn's `MetricFrame` and built-in group metrics to make the gap visible, add uncertainty,
and write a decision — not a dashboard.

This is a **measurement** lab, not a legal or ethical verdict.

In [ ]:
# --- Workshop bootstrap: run this cell first ------------------------------------
# JupyterLab starts every kernel inside the notebook's own folder. Move to the
# toolkit root so shared modules (demo_agent, workshop_utils) import and the
# _evidence/ output paths resolve, no matter where Jupyter was launched from.
import os, sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "workshop_utils.py").exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Workshop root:", ROOT)

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score

from workshop_utils import require_package, save_json

require_package("fairlearn")
from fairlearn.metrics import (
    MetricFrame, count, selection_rate, false_positive_rate, false_negative_rate,
    demographic_parity_difference, equalized_odds_difference,
)

## 1. Synthetic data with a designed disparity

`y_true` = the ticket really needed escalation. `y_pred` = the classifier's call. English gets ~8%
random errors; Hindi gets a much higher **false-negative** rate (urgent tickets missed) plus some
false positives.

In [ ]:
rng = np.random.default_rng(17)
n_per_group = 240
language = np.array(["English"] * n_per_group + ["Hindi"] * n_per_group)
y_true = rng.binomial(1, 0.40, size=2 * n_per_group)
y_pred = y_true.copy()

english_idx = np.where(language == "English")[0]
flip_en = rng.choice(english_idx, size=20, replace=False)
y_pred[flip_en] = 1 - y_pred[flip_en]

hindi_idx = np.where(language == "Hindi")[0]
hindi_pos = hindi_idx[y_true[hindi_idx] == 1]
hindi_neg = hindi_idx[y_true[hindi_idx] == 0]
missed = rng.choice(hindi_pos, size=max(1, int(0.38 * len(hindi_pos))), replace=False)
false_alarms = rng.choice(hindi_neg, size=max(1, int(0.12 * len(hindi_neg))), replace=False)
y_pred[missed] = 0
y_pred[false_alarms] = 1

df = pd.DataFrame({"language": language, "y_true": y_true, "y_pred": y_pred})
print("Overall accuracy:", round(accuracy_score(df.y_true, df.y_pred), 3), "  <- looks fine")
df.groupby("language").size().rename("count").to_frame()

## 2. `MetricFrame`: the same metrics, disaggregated

Fairlearn ships the group-metric functions (`selection_rate`, `false_positive_rate`,
`false_negative_rate`, `count`). `by_group` makes disaggregation explicit; `difference()` and
`ratio()` summarise disparity — read them alongside base rates, sample sizes, and uncertainty.

In [ ]:
metrics = {
    "count": count,
    "accuracy": accuracy_score,
    "selection_rate": selection_rate,
    "false_positive_rate": false_positive_rate,
    "false_negative_rate": false_negative_rate,
}
frame = MetricFrame(metrics=metrics, y_true=df["y_true"], y_pred=df["y_pred"], sensitive_features=df["language"])

display(frame.overall.to_frame("overall"))
display(frame.by_group)
display(pd.DataFrame({"difference": frame.difference(), "ratio": frame.ratio()}))

In [ ]:
dpd = demographic_parity_difference(df.y_true, df.y_pred, sensitive_features=df.language)
eod = equalized_odds_difference(df.y_true, df.y_pred, sensitive_features=df.language)
print(f"Demographic parity difference: {dpd:.3f}")
print(f"Equalized odds difference:     {eod:.3f}   <- driven by the FNR gap")

## 3. Add uncertainty with a transparent bootstrap

Small groups and rare outcomes produce unstable rates. The bootstrap here is illustrative; choose an
uncertainty method appropriate to the metric and sampling design.

In [ ]:
def bootstrap_metric(group_df, metric_fn, rounds=500, seed=23):
    local_rng = np.random.default_rng(seed)
    values = []
    for _ in range(rounds):
        sample = group_df.iloc[local_rng.integers(0, len(group_df), len(group_df))]
        values.append(metric_fn(sample.y_true, sample.y_pred))
    return np.quantile(values, [0.025, 0.5, 0.975])

intervals = []
for group, group_df in df.groupby("language"):
    lo, mid, hi = bootstrap_metric(group_df, false_negative_rate)
    intervals.append({"language": group, "metric": "false_negative_rate", "lower": lo, "median": mid, "upper": hi, "n": len(group_df)})
intervals_df = pd.DataFrame(intervals)
intervals_df

In [ ]:
by_group = frame.by_group
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.bar(by_group.index, by_group["false_negative_rate"], color=["#4c72b0", "#dd8452"])
for i, row in intervals_df.iterrows():
    ax.errorbar(row["language"], row["median"], yerr=[[row["median"] - row["lower"]], [row["upper"] - row["median"]]], fmt="none", ecolor="black", capsize=4)
ax.set_ylabel("False-negative rate (95% bootstrap CI)")
ax.set_title("Missed escalations by interaction language")
plt.tight_layout(); plt.show()

## 4. Decision, not dashboard theatre

The synthetic Hindi false-negative rate is materially worse: urgent tickets are more likely to be
missed. A reasonable response might be to constrain launch, improve language-specific data and labels,
check upstream speech/OCR errors, adjust workflow or thresholds with domain owners, and add human review
— then re-measure **all** error types and utility. Fairlearn's `ThresholdOptimizer` / reductions are the
next step once you have per-group scores rather than hard labels.

In [ ]:
assessment = {
    "use_case": "synthetic support-ticket escalation",
    "groups": ["English", "Hindi"],
    "sample_sizes": df.groupby("language").size().to_dict(),
    "overall": frame.overall.to_dict(),
    "by_group": by_group.reset_index().to_dict(orient="records"),
    "difference": frame.difference().to_dict(),
    "ratio": frame.ratio().to_dict(),
    "demographic_parity_difference": float(dpd),
    "equalized_odds_difference": float(eod),
    "uncertainty": intervals_df.to_dict(orient="records"),
    "decision": "Do not ship as one undifferentiated workflow; investigate and mitigate Hindi false negatives.",
    "owner": "TBD in production",
    "caveats": [
        "synthetic data", "groups are not exhaustive", "metrics do not determine legality or ethics",
        "deployment workflow and error costs require domain review",
    ],
}
out = save_json("_evidence/09_fairness_assessment.json", assessment)
assert by_group.loc["Hindi", "false_negative_rate"] > by_group.loc["English", "false_negative_rate"]
assert eod > 0.15, "the designed disparity should be visible in equalized-odds difference"
print("PASS: aggregate metrics did not hide the designed subgroup failure")
print("Wrote", out.resolve())